In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def make_phenotype_file(data):
    '''
    Create a phenotype file from the given data.
    Inputs:
        data: pd.DataFrame - Raw data containing phenotype information  
    Outputs:
        None - Saves the phenotype data to 'phenotype_file.csv'
    '''
    phenotype_data = data.iloc[0:52, 0:318]
    phenotype_data.to_csv('../Phenotype/phenotype_file.csv', index=False)
    return

def make_proteomics_file(data):
    '''
    Create a proteomics file from the given data.'
    Inputs:
        data: pd.DataFrame - Raw data containing proteomics information
    Outputs:
        None - Saves the proteomics data to 'proteomics_file.csv'
    '''
    proteomics_data = pd.concat([data.iloc[0:12, 0:318], data.iloc[52:32497, 0:318]])
    proteomics_data.to_csv('../Proteomics_datasets/proteomics_file.csv', index=False)
    return

def make_transcriptomics_file(data):
    '''
    Create a transcriptomics file from the given data.
    Inputs:
        data: pd.DataFrame - Raw data containing transcriptomics information
    Outputs:
        None - Saves the transcriptomics data to 'transcriptomics_file.csv'
    '''
    transcriptomics_data = pd.concat([data.iloc[0:12, 0:318], data.iloc[32497:75817, 0:318]])
    transcriptomics_data.to_csv('/Transciptomics_datasets/transcriptomics_file.csv', index=False)
    return

def make_metabolomics_file(data):
    '''
    Create a metabolomics file from the given data.
    Inputs:
        data: pd.DataFrame - Raw data containing metabolomics information
    Outputs:
        None - Saves the metabolomics data to 'metabolomics_file.csv'
    '''
    metabolomics_data = pd.concat([data.iloc[0:12, 0:318], data.iloc[75817:76573, 0:318]])
    metabolomics_data.to_csv('../Metabolomics_datasets/metabolomics_file.csv', index=False)
    return

In [ ]:
data = pd.read_csv('../aData_S1_AllOmicsandPhenotypeData.csv')
make_metabolomics_file(data)
make_proteomics_file(data)
make_transcriptomics_file(data)
make_phenotype_file(data)

In [ ]:
def preprocess_data_metabolomics(data, header_row=7, metadata_rows=12):
    '''
    Preprocess metabolomics data by handling metadata, converting to numeric,
    grouping by the first column, and filling missing values.
    Inputs:
        data: pd.DataFrame - Raw data with metadata and measurements
        header_row: int - Row index to use as header
        metadata_rows: int - Number of metadata rows to skip
    Outputs:
        pd.DataFrame - Preprocessed data ready for analysis
    '''
    df = data.copy()
    df.columns = df.iloc[header_row]
    df_filtered = df.drop(index=range(metadata_rows)).reset_index(drop=True)
    df_filtered = df_filtered.iloc[:, 2:].reset_index(drop=True)
    df_filtered.columns = df_filtered.columns.str.strip()
    group_col = df_filtered.columns[0]
    feature_cols = df_filtered.columns[1:] 
    numeric_data = df_filtered[feature_cols].apply(pd.to_numeric, errors='coerce')
    def smart_mean(series):
        if series.notna().sum() == 0:
            return np.nan
        elif series.notna().sum() == 1:
            return series.dropna().iloc[0]
        else:
            return series.mean()
    grouping_keys = df_filtered[group_col]
    combined_data = numeric_data.groupby(grouping_keys).agg(smart_mean)
    np.random.seed(42)
    combined_data = combined_data.apply(lambda row: row.fillna(row.median()+np.random.normal(0, 0.05)), axis=1)
    combined_data = combined_data.reset_index()
    return combined_data

In [ ]:
def get_diet(data_metabolomics, row = 4):
    '''
    Extract diet information from the metabolomics data.
    Inputs:
        data_metabolomics: pd.DataFrame - Raw metabolomics data with diet info
        row: int - Row index where diet information is located
    Outputs:
        np.ndarray - Array of diet information
    '''
    diet_info = data_metabolomics.iloc[row, 2:]
    diet_info = diet_info.reset_index(drop=True)
    diet_array = np.array(diet_info)
    diet_array = diet_array[1:]
    print(diet_array)
    return diet_array  

In [ ]:
data_metabolomics = pd.read_csv('../Metabolomics_datasets/metabolomics_file.csv')
processed_metabolomics = preprocess_data_metabolomics(data_metabolomics)
processed_metabolomics.to_csv('../Metabolomics_datasets/process_met_file.csv', index=False)
data_diet = pd.read_csv('../Metabolomics_datasets/metabolomics_file.csv')
data_metabolomics = pd.read_csv('../Metabolomics_datasets/process_met_file.csv')

In [ ]:
def preprocess_data_transcriptomics(data, header_row=7, metadata_rows=12, name_col=2):
    '''
    Preprocess transcriptomics data by handling metadata, converting to numeric,
    and removing samples with missing data.

    Inputs:
        data: pd.DataFrame - Raw data with metadata and measurements
        header_row: int - Row index to use as header
        metadata_rows: int - Number of metadata rows to skip

    Outputs:
        pd.DataFrame - Preprocessed data ready for analysis, including feature ID column
    '''
    df = data.copy()
    df.columns = df.iloc[header_row]
    df_filtered = df.drop(index=range(metadata_rows)).reset_index(drop=True)
    df_filtered = df_filtered.iloc[:, name_col:].reset_index(drop=True)
    df_filtered.columns = df_filtered.columns.str.strip()
    second_col_name = df_filtered.columns[0]
    second_col = df_filtered[second_col_name]
    feature_cols = df_filtered.columns[1:]
    numeric_data = df_filtered[feature_cols].apply(pd.to_numeric, errors='coerce')
    combined_data = pd.concat([second_col, numeric_data], axis=1)
    combined_data = combined_data.dropna(axis=1, how='any')

    return combined_data



In [ ]:
not_preprocessed_transcriptomics = pd.read_csv('../Transcriptomics_datasets/transcriptomics_file.csv') # reads the raw transcriptomics data
processed_transcriptomics = preprocess_data_transcriptomics(not_preprocessed_transcriptomics) # preprocesses the transcriptomics data
processed_transcriptomics.to_csv('../Transcriptomics_datasets/process_trans_file.csv', index=False) # saves the preprocessed transcriptomics data
data_transcriptomics = pd.read_csv('../Transcriptomics_datasets/process_trans_file.csv') # reads the preprocessed transcriptomics data

In [ ]:
not_preprocessed_transcriptomics = pd.read_csv('../Transcriptomics_datasets/transcriptomics_file.csv') # reads the raw transcriptomics data
processed_transcriptomics = preprocess_data_transcriptomics(not_preprocessed_transcriptomics, name_col=1) # preprocesses the transcriptomics data with the second column as feature ID column
processed_transcriptomics.to_csv('../Transcriptomics_datasets/process_trans_file_1_col.csv', index=False) # saves the preprocessed transcriptomics data with the second column as feature ID column
data_transcriptomics = pd.read_csv('../Transcriptomics_datasets/process_trans_file_1_col.csv') # reads the preprocessed transcriptomics data with the second column as feature ID column

In [ ]:
def preprocess_data_proteomics(data, header_row=7, metadata_rows=12):
    '''
    Preprocess proteomics data by handling metadata, converting to numeric,
    removing metadata rows, and averaging rows with the same identifier.

    Inputs:
        data: pd.DataFrame - Raw data with metadata and measurements
        header_row: int - Row index to use as header
        metadata_rows: int - Number of metadata rows to skip

    Outputs:
        combined_data: pd.DataFrame - Preprocessed data with unique feature IDs
                                     and averaged values
    '''
    df = data.copy()
    df.columns = df.iloc[header_row]
    df_filtered = df.drop(index=range(metadata_rows)).reset_index(drop=True)
    feature_id_col = df_filtered.iloc[:, 2]
    numeric_data = df_filtered.iloc[:, 3:].apply(pd.to_numeric, errors='coerce')
    combined_data = pd.concat([feature_id_col, numeric_data], axis=1)
    combined_data = combined_data.groupby(combined_data.columns[0]).mean().reset_index()
    
    return combined_data


In [ ]:
data_proteomics = pd.read_csv('../Proteomics_datasets/proteomics_file.csv')
processed_proteomics = preprocess_data_proteomics(data_proteomics)
processed_proteomics.to_csv('../Proteomics_datasets/process_prot_file.csv', index=False)

In [ ]:
def combine_transcriptomics_isoforms(data, header_row=7, metadata_rows=12):
    '''
    Unify transcriptomics data by combining isoforms into genes, averaging values for the same gene ID.

    Inputs:
        data: pd.DataFrame - Raw data with metadata and measurements
        header_row: int - Row index to use as header
        metadata_rows: int - Number of metadata rows to skip

    Outputs:
        combined_data: pd.DataFrame - Preprocessed data with unique feature IDs
                                     and averaged values
    '''
    df = data.copy()
    df.columns = df.iloc[header_row]
    df_filtered = df.drop(index=range(metadata_rows)).reset_index(drop=True)
    feature_id_col = df_filtered.iloc[:, 2]
    numeric_data = df_filtered.iloc[:, 3:].apply(pd.to_numeric, errors='coerce')
    combined_data = pd.concat([feature_id_col, numeric_data], axis=1)
    combined_data = combined_data.groupby(combined_data.columns[0]).mean().reset_index()
    combined_data = combined_data.dropna(axis=1, how='any')
    return combined_data

In [ ]:
data_transcriptomics = pd.read_csv('../Transcriptomics_datasets/transcriptomics_file.csv')
data_transcriptomics = combine_transcriptomics_isoforms(data_transcriptomics, header_row=7, metadata_rows=12)
data_transcriptomics.to_csv('../Transcriptomics_datasets/process_trans_combinedisoforms.csv', index=False)

In [ ]:
def preprocess_data_serumphenotype(data, header_row=7, metadata_rows=12):
    '''
    Preprocess serum phenotype data by handling metadata, converting to numeric,
    and removing metadata rows.

    Inputs:
        data: pd.DataFrame - Raw data with metadata and measurements
        header_row: int - Row index to use as header
        metadata_rows: int - Number of metadata rows to skip

    Outputs:
        combined_data: pd.DataFrame - Preprocessed data with unique feature IDs
                                     and averaged values
    '''
    df = data.copy()
    df.columns = df.iloc[header_row]
    df_filtered = df.drop(index=range(metadata_rows)).reset_index(drop=True)
    feature_id_col = df_filtered.iloc[:, 2]
    numeric_data = df_filtered.iloc[:, 3:].apply(pd.to_numeric, errors='coerce')
    combined_data = pd.concat([feature_id_col, numeric_data], axis=1)
    combined_data = combined_data.dropna(axis=1, how='any')
    
    return combined_data 

In [ ]:
phenotype_serum = pd.read_csv('../Phenotype/phenotype_file.csv')
processed_phenotype_serum = preprocess_data_serumphenotype(phenotype_serum, header_row=7, metadata_rows=33)
processed_phenotype_serum.to_csv('../Phenotype/process_phenotype_serum_file.csv', index=False)

In [ ]:
def remove_low_expression_genes(data):
    '''
    Remove genes (rows) where more than half of the values are zero.

    Inputs:
        data: pd.DataFrame - Data with genes as rows and samples as columns

    Outputs:
        pd.DataFrame - Filtered data with genes having >50% zeros removed
    '''
    numeric_data = data.iloc[:, 1:]
    zero_counts = (numeric_data == 0).sum(axis=1)
    keep_mask = zero_counts <= (numeric_data.shape[1] / 2)
    filtered_data = data[keep_mask].reset_index(drop=True)
    return filtered_data

In [ ]:
data_transcriptomics = pd.read_csv('../Transcriptomics_datasets/process_trans_combinedisoforms.csv')
process_trans_removelowexp = remove_low_expression_genes(data_transcriptomics)
process_trans_removelowexp.to_csv('../Transcriptomics_datasets/process_trans_removelowexp_combiso.csv', index=False)

In [ ]:
data_transcriptomics = pd.read_csv('../Transcriptomics_datasets/process_trans_file.csv')
process_trans_removelowexp = remove_low_expression_genes(data_transcriptomics)
process_trans_removelowexp.to_csv('../Transcriptomics_datasets/process_trans_removelowexp.csv', index=False)